In [0]:
%run ../common/common_logging

In [0]:
%run ../common/common_config_loader

In [0]:
# Initialize configuration and logging
config = get_config()
logger = get_logger(__name__)

logger.info("Bronze Job Snapshot Writer initialized")

# Bronze Job Snapshot Writer

Captures immutable snapshots of each ingestion job execution.

**Purpose**: Audit trail for all Bronze ingestion jobs  
**Pattern**: Append-only job metadata  
**No transformations**: Raw job state only

In [0]:
# Job execution parameters
dbutils.widgets.text("job_id", "", "Job ID")
dbutils.widgets.text("job_name", "", "Job Name")
dbutils.widgets.text("batch_id", "", "Batch ID")
dbutils.widgets.text("run_id", "", "Run ID")
dbutils.widgets.dropdown("status", "running", ["running", "success", "failed", "partial"], "Status")
dbutils.widgets.text("config_json", "{}", "Configuration JSON")

# Get parameter values
job_id = dbutils.widgets.get("job_id")
job_name = dbutils.widgets.get("job_name")
batch_id = dbutils.widgets.get("batch_id")
run_id = dbutils.widgets.get("run_id")
status = dbutils.widgets.get("status")
config_json = dbutils.widgets.get("config_json")

# Use configuration from common config
catalog = config.CATALOG
schema = config.BRONZE_SCHEMA

# Validate required parameters
if not job_id:
    raise ValueError("job_id is required")
if not job_name:
    raise ValueError("job_name is required")
if not batch_id:
    raise ValueError("batch_id is required")
if not run_id:
    raise ValueError("run_id is required")

# Log parameters
log_metrics(logger, {
    "Job ID": job_id,
    "Job Name": job_name,
    "Batch ID": batch_id,
    "Run ID": run_id,
    "Status": status
}, prefix="Job Parameters")

In [0]:
%sql
-- Create job snapshot table if not exists
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.bronze_job_snapshot (
  snapshot_id STRING,
  job_id STRING,
  job_name STRING,
  batch_id STRING,
  run_id STRING,
  status STRING,
  config_json STRING,
  snapshot_timestamp TIMESTAMP,
  ingestion_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Immutable snapshots of Bronze ingestion job executions'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true'
);

In [0]:
from pyspark.sql import functions as F
from uuid import uuid4
from datetime import datetime, timezone

log_section_start(logger, "Writing Job Snapshot", width=80)

try:
    # Create snapshot record
    snapshot_id = str(uuid4())
    current_timestamp = datetime.now(timezone.utc)
    
    snapshot_df = spark.createDataFrame([(
        snapshot_id,
        job_id,
        job_name,
        batch_id,
        run_id,
        status,
        config_json,
        current_timestamp,
        current_timestamp
    )], schema="snapshot_id STRING, job_id STRING, job_name STRING, batch_id STRING, run_id STRING, status STRING, config_json STRING, snapshot_timestamp TIMESTAMP, ingestion_timestamp TIMESTAMP")
    
    # Build target table name using config
    target_table = config.get_bronze_table("bronze_job_snapshot")
    
    logger.info(f"Writing snapshot to {target_table}")
    
    # Append to Bronze table
    snapshot_df.write.mode("append").option("mergeSchema", "true").saveAsTable(target_table)
    
    # Log success
    logger.info(f"✓ Job snapshot written successfully")
    log_metrics(logger, {
        "Snapshot ID": snapshot_id,
        "Job Name": job_name,
        "Job ID": job_id,
        "Batch ID": batch_id,
        "Run ID": run_id,
        "Status": status,
        "Target Table": target_table
    }, prefix="Snapshot Details")
    
except Exception as e:
    logger.error(f"Failed to write job snapshot: {str(e)}")
    log_exception(logger, e, "During job snapshot write")
    raise

finally:
    log_section_end(logger, "Job Snapshot Write", width=80)